# Allergen Detection from Ingredient Lists
## Multi-label Classification with MobileBERT

This notebook trains a multi-label classifier to detect 8 allergens from product ingredient texts.

## Overview

Trains MobileBERT model for multi-label allergen classification.

## Workflow

1. Load and prepare data using utility modules
2. Split into train/val/test with stratified splitting
3. Train model with weighted loss
4. Evaluate and compute optimal thresholds
5. Save final model

## 1. Setup & Imports

Using modular utility functions to reduce code duplication

In [11]:
import os
import random
import shutil
import warnings
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

warnings.filterwarnings('ignore')

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import f1_score
from scipy.special import expit

# Import project utility modules
from utils.data_utils import (
    get_data_directories,
    load_labeled_data,
    create_stratified_splits,
    augment_dataframe
)
from utils.text_processing import (
    clean_html,
    get_allergen_list,
    allergens_to_binary
)
from utils.model_utils import (
    compute_class_weights,
    find_best_thresholds,
    apply_thresholds
)
from utils.evaluation import (
    print_classification_report,
    print_per_class_metrics,
    error_analysis
)

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("All imports completed using utility modules.")

All imports completed using utility modules.


## 2. Load and Prepare Data

In [12]:
# Get project directories for consistent path handling
dirs = get_data_directories()
print(f"Project base directory: {dirs['base']}")

# Load labeled data using utility function
# Use the enhanced dataset which contains detected allergens
df = load_labeled_data(filepath=os.path.join(dirs['final'], 'labeled_dataset_enhanced.csv'))
print(f"Dataset shape: {df.shape}")
df.head()

Project base directory: /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION
Dataset shape: (1057, 11)


,code,brands,product_name_en,ingredients_text_en,detected_allergens,official_allergens_mapped,traces_allergens,consensus_allergens,combined_allergens,may_contain,coconut
0,8850389100684,Sappe Public Company Limited,Mogu litchi,"water, lychee juice from concentrate 25%, coco...",[],[],[],[],"{'detected_only': [], 'detected_or_official': ...",[],['coconut']
1,4800194105958,Oishi,Oishi Prawn Crackers (Sweet Extra Flavor),"wheat, tapioca starch, vegetable oil (may cons...",['wheat'],[],[],[],"{'detected_only': ['wheat'], 'detected_or_offi...",[],['coconut']
2,4806533726822,Gardenia bakeries (Phils) Inc,Neu Bake,"high protein wheat flour (with vitamins b1, b2...","['milk', 'wheat']",[],[],[],"{'detected_only': ['milk', 'wheat'], 'detected...",[],['coconut']
3,4800361316934,Maggi,Savor Liquid Seasoning Chilimansi,"water, lodized salt, sugar, flavor enhancers (...",['wheat'],[],[],[],"{'detected_only': ['wheat'], 'detected_or_offi...",[],[]
4,4807770273704,Lucky Me,lm pc original 80g,"wheat flour, vegetable oil (palm oil, green te...","['soy', 'wheat']",[],[],[],"{'detected_only': ['soy', 'wheat'], 'detected_...",[],['coconut']


In [13]:
# Clean HTML tags from ingredients using utility function
df["ingredients_cleaned"] = df["ingredients_text_en"].apply(clean_html)

# Convert detected allergens list to binary vector
df["labels"] = df["detected_allergens"].apply(allergens_to_binary)

print(f"Sample cleaned text: {df['ingredients_cleaned'].iloc[0][:200]}")
print(f"Label distribution (per class): {np.array(df['labels'].tolist()).sum(axis=0)}")

Sample cleaned text: water, lychee juice from concentrate 25%, coconut 25%, fructose, sucrose, acidifiers (citric acid, calcium lactate), lychee flavor, preservative e211, gelling agent: gellan gum, color: e129 - 0.0001% 
Label distribution (per class): [497  98  88  16 278 355  77  32]


## 3. Train/Validation/Test Split (Stratified)

Using utility function for multilabel stratified split

In [14]:
# Prepare data for splitting
X = df["ingredients_cleaned"].values
y = np.array(df["labels"].tolist())

# Create stratified splits using utility function
# Using 70% train, 15% validation, 15% test
train_texts, val_texts, test_texts, train_labels, val_labels, test_labels = create_stratified_splits(
    X, y,
    train_size=0.7,
    val_size=0.15,
    test_size=0.15,
    random_state=SEED
)

print(f"Train size: {len(train_texts)} (positive samples: {np.array(train_labels).sum(axis=0).sum()})")
print(f"Val size:   {len(val_texts)}   (positive samples: {np.array(val_labels).sum(axis=0).sum()})")
print(f"Test size:  {len(test_texts)}  (positive samples: {np.array(test_labels).sum(axis=0).sum()})")

Train size: 739 (positive samples: 1007)
Val size:   159   (positive samples: 216)
Test size:  159  (positive samples: 218)


## 4. Data Augmentation (Training Set Only)

Using utility function for data augmentation

In [15]:
# Create temporary dataframes for augmentation
train_df = pd.DataFrame({
    'text': train_texts,
    'labels': train_labels.tolist(),
    'ingredients_cleaned': train_texts  # Simplified for augmentation
})

# Apply data augmentation using utility function
train_aug_df = augment_dataframe(train_df, num_augmented=2)
print(f"Original training size: {len(train_df)}")
print(f"Augmented training size: {len(train_aug_df)}")

# Combine original training + augmented training
combined_train_df = pd.concat([train_df, train_aug_df], ignore_index=True)
print(f"Combined training size: {len(combined_train_df)}")

# Extract texts and labels for training
train_texts_final = combined_train_df["text"].tolist()
train_labels_final = np.array(combined_train_df["labels"].tolist())

# Validation and test sets remain unchanged
val_texts_final = val_texts
val_labels_final = np.array(val_labels)
test_texts_final = test_texts
test_labels_final = np.array(test_labels)

Original training size: 739
Augmented training size: 1211
Combined training size: 1950


## 5. Compute Class Weights for Imbalance

Using utility function to compute class weights

In [16]:
# Compute class weights using utility function
class_weights_tensor = compute_class_weights(train_labels_final)

print("Class weights:")
allergen_list = get_allergen_list()
for i, allergen in enumerate(allergen_list):
    print(f"  {allergen:10s}: {class_weights_tensor[i]:.3f}")

Class weights:
  milk      : 0.371
  eggs      : 1.011
  peanuts   : 1.064
  tree_nuts : 1.858
  soy       : 0.575
  wheat     : 0.481
  fish      : 1.126
  shellfish : 1.514


## 6. Tokenization

Determine optimal max_length based on 95th percentile of token lengths

In [17]:
# Detect device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_name = "google/mobilebert-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Compute max length from training texts
sample_lengths = [len(tokenizer.encode(t, truncation=False)) for t in train_texts_final[:500]]
max_len = int(np.percentile(sample_lengths, 95))
MAX_LEN = min(max_len, 256)
print(f"Using max_length = {MAX_LEN}")

def tokenize_data(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

train_encodings = tokenize_data(train_texts_final)
val_encodings = tokenize_data(val_texts_final)
test_encodings = tokenize_data(test_texts_final)

class FoodDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item
    
    def __len__(self):
        return len(self.labels)

train_dataset = FoodDataset(train_encodings, train_labels_final)
val_dataset = FoodDataset(val_encodings, val_labels_final)
test_dataset = FoodDataset(test_encodings, test_labels_final)

# Ensure training output directory exists using dirs for consistency
output_dir = os.path.join(dirs['base'], "models", "mobilebert_allergen")
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory ready: {output_dir}")

Using device: cuda
Using max_length = 221
Output directory ready: /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/models/mobilebert_allergen


## 7. Model Definition & Training Utilities

Using manual PyTorch training loop (HuggingFace Trainer causes segfault
in this environment's PyTorch 2.12.0+cu130 build).

In [18]:
# Load model
n_classes = len(get_allergen_list())
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=n_classes,
    problem_type="multi_label_classification"
)
model = model.to(device)
print(f"Model loaded on {device}")

class_weights_tensor = class_weights_tensor.to(device)

def run_train_epoch(model, dataloader, optimizer, scheduler, loss_fn, device):
    """Run one training epoch, returns average loss."""
    model.train()
    total_loss = 0.0
    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = loss_fn(outputs.logits, batch["labels"])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)


def evaluate(model, dataloader, loss_fn, device):
    """Evaluate model, returns (avg_loss, macro_f1, logits_array, labels_array)."""
    model.eval()
    total_loss = 0.0
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = loss_fn(outputs.logits, batch["labels"])
            total_loss += loss.item()
            all_logits.append(outputs.logits.cpu().numpy())
            all_labels.append(batch["labels"].cpu().numpy())
    logits = np.concatenate(all_logits)
    labels = np.concatenate(all_labels)
    probs = expit(logits)
    preds = (probs > 0.5).astype(int)
    f1 = f1_score(labels, preds, average="macro", zero_division=0)
    return total_loss / len(dataloader), f1, logits, labels


def predict(model, dataset, device, batch_size=8):
    """Run inference and return (logits, labels) arrays."""
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            all_logits.append(outputs.logits.cpu().numpy())
            all_labels.append(batch["labels"].cpu().numpy())
    return np.concatenate(all_logits), np.concatenate(all_labels)


print("Training utilities ready.")

Some weights of MobileBertForSequenceClassification were not initialized from the model checkpoint at google/mobilebert-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded on cuda
Training utilities ready.


## 8. Training Configuration

Hyperparameters for manual training loop.

In [19]:
# Training hyperparameters
batch_size = 8
num_epochs = 20
learning_rate = 2e-5
weight_decay = 0.01
max_grad_norm = 1.0
early_stopping_patience = 3

train_steps_per_epoch = max(1, len(train_texts_final) // batch_size)
total_training_steps = train_steps_per_epoch * num_epochs
warmup_steps = int(0.1 * total_training_steps)

print(f"Samples: {len(train_texts_final)} train, {len(val_texts_final)} val")
print(f"Steps per epoch: {train_steps_per_epoch}")
print(f"Total training steps: {total_training_steps}, warmup: {warmup_steps}")

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Loss function with class weights
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=class_weights_tensor, reduction='mean')

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# LR scheduler with linear warmup + linear decay
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

print("Training configuration ready.")

Samples: 1950 train, 159 val
Steps per epoch: 243
Total training steps: 4860, warmup: 486
Training configuration ready.


In [20]:
%%time
print("=" * 60)
print("Starting training")
print("=" * 60)

best_val_f1 = 0.0
best_model_state = None
no_improve_epochs = 0
train_losses, val_losses, val_f1s = [], [], []

output_dir = os.path.join(dirs['base'], "models", "mobilebert_allergen")
os.makedirs(output_dir, exist_ok=True)

for epoch in range(1, num_epochs + 1):
    # Train for one epoch
    train_loss = run_train_epoch(model, train_loader, optimizer, scheduler, loss_fn, device)
    
    # Evaluate on validation set
    val_loss, val_f1, val_logits, val_labels = evaluate(model, val_loader, loss_fn, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_f1s.append(val_f1)
    
    # Early stopping check
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve_epochs = 0
        # Save checkpoint for best model
        torch.save(model.state_dict(), os.path.join(output_dir, "best_model.pt"))
        tokenizer.save_pretrained(output_dir)
    else:
        no_improve_epochs += 1
    
    print(f"Epoch {epoch:2d}/{num_epochs}  "
          f"Train Loss: {train_loss:.4f}  "
          f"Val Loss: {val_loss:.4f}  "
          f"Val Macro F1: {val_f1:.4f}  "
          f"{'*' if val_f1 == best_val_f1 and epoch > 1 else ''}"
          f"{' (no improv. ' + str(no_improve_epochs) + ')' if no_improve_epochs > 0 else ''}")
    
    if no_improve_epochs >= early_stopping_patience:
        print(f"\nEarly stopping triggered after {epoch} epochs.")
        break

# Restore best model
if best_model_state is not None:
    model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})
    model = model.to(device)

print(f"\nTraining complete. Best Val Macro F1: {best_val_f1:.4f}")
print(f"Training history saved in train_losses, val_losses, val_f1s arrays")

Starting training
Epoch  1/20  Train Loss: 1383257.4554  Val Loss: 17.1012  Val Macro F1: 0.0503  
Epoch  2/20  Train Loss: 93.0228  Val Loss: 0.1293  Val Macro F1: 0.5639  *
Epoch  3/20  Train Loss: 0.1151  Val Loss: 0.0859  Val Macro F1: 0.6881  *
Epoch  4/20  Train Loss: 0.0522  Val Loss: 0.0834  Val Macro F1: 0.7754  *
Epoch  5/20  Train Loss: 0.0383  Val Loss: 0.0737  Val Macro F1: 0.8155  *
Epoch  6/20  Train Loss: 0.0246  Val Loss: 0.0864  Val Macro F1: 0.7974   (no improv. 1)
Epoch  7/20  Train Loss: 0.0194  Val Loss: 0.0977  Val Macro F1: 0.8130   (no improv. 2)
Epoch  8/20  Train Loss: 0.0158  Val Loss: 0.0904  Val Macro F1: 0.8020   (no improv. 3)

Early stopping triggered after 8 epochs.

Training complete. Best Val Macro F1: 0.8155
Training history saved in train_losses, val_losses, val_f1s arrays
CPU times: user 4min 9s, sys: 34.8 s, total: 4min 44s
Wall time: 4min 44s


## 9. Evaluate on Validation Set (Baseline with 0.5 threshold)

In [21]:
# Use manual predict instead of trainer.predict
val_logits, val_output_labels = predict(model, val_dataset, device, batch_size=batch_size)
val_probs = expit(val_logits)
val_preds_05 = (val_probs >= 0.5).astype(int)

print("Baseline (threshold=0.5) Macro F1:",
      f1_score(val_output_labels, val_preds_05, average="macro", zero_division=0))

Baseline (threshold=0.5) Macro F1: 0.815473438844645


## 10. Per-Class Threshold Optimization

We find optimal thresholds on validation set to maximize macro F1 per class
Using utility function for threshold optimization

In [22]:
# Find optimal thresholds using utility function
best_thresholds = find_best_thresholds(val_probs, val_labels_final)
np.save(os.path.join(dirs['models'], "best_thresholds.npy"), best_thresholds)

print("Optimal thresholds:")
allergen_list = get_allergen_list()
for i, allergen in enumerate(allergen_list):
    print(f"{allergen:12s} → threshold={best_thresholds[i]:.2f}")

Optimal thresholds:
milk         → threshold=0.24
eggs         → threshold=0.95
peanuts      → threshold=0.84
tree_nuts    → threshold=0.01
soy          → threshold=0.92
wheat        → threshold=0.82
fish         → threshold=0.96
shellfish    → threshold=0.40


## 11. Apply Optimized Thresholds

Using utility function to apply thresholds

In [23]:
# Apply optimized thresholds using utility function
val_preds_opt = apply_thresholds(val_probs, best_thresholds)

print("\nValidation set after threshold tuning:")
print(f"Macro F1: {f1_score(val_labels_final, val_preds_opt, average='macro', zero_division=0):.4f}")
print(f"Micro F1: {f1_score(val_labels_final, val_preds_opt, average='micro', zero_division=0):.4f}")


Validation set after threshold tuning:
Macro F1: 0.8696
Micro F1: 0.9390


## 12. Final Evaluation on Test Set

Using utility functions for evaluation and error analysis

In [24]:
# Use manual predict instead of trainer.predict
test_logits, test_labels_true = predict(model, test_dataset, device, batch_size=batch_size)
test_probs = expit(test_logits)
test_preds = apply_thresholds(test_probs, best_thresholds)

print("\n=== Test Set Performance ===")
print_classification_report(test_labels_true, test_preds, get_allergen_list(), prefix="Test Set")

# Optional: Per-class metrics
print_per_class_metrics(test_labels_true, test_preds, get_allergen_list(), prefix="Per-class metrics on test set:")

# Optional: Error analysis
error_indices = error_analysis(test_texts_final, test_labels_true, test_preds, get_allergen_list(), max_errors=3)


=== Test Set Performance ===
=== Test Set ===
              precision    recall  f1-score   support

        milk     0.9722    0.9333    0.9524        75
        eggs     1.0000    1.0000    1.0000        14
     peanuts     1.0000    0.9231    0.9600        13
   tree_nuts     0.2222    0.6667    0.3333         3
         soy     0.9737    0.8810    0.9250        42
       wheat     1.0000    0.9444    0.9714        54
        fish     1.0000    0.5833    0.7368        12
   shellfish     0.8000    0.8000    0.8000         5

   micro avg     0.9471    0.9037    0.9249       218
   macro avg     0.8710    0.8415    0.8349       218
weighted avg     0.9701    0.9037    0.9315       218
 samples avg     0.6431    0.6251    0.6279       218

Macro F1: 0.8349
Micro F1: 0.9249


Per-class metrics on test set:
milk          Prec: 97.22%  Rec: 93.33%  F1: 95.24%
eggs          Prec: 100.00%  Rec: 100.00%  F1: 100.00%
peanuts       Prec: 100.00%  Rec: 92.31%  F1: 96.00%
tree_nuts     Prec: 2

## 13. Save Final Model

Saving model and tokenizer

In [ ]:
final_dir = os.path.join(dirs['models'], "mobilebert_allergen_final")
if os.path.exists(final_dir):
    shutil.rmtree(final_dir)

# Save model and tokenizer manually (not via trainer.save_model)
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"Model saved to {final_dir}")

Model saved to /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/models/mobilebert_allergen_final


: 